# Experiment 2 — Single-Chain Mode Coverage on a CIFAR-10 EBM

Runs **Score-Repellent Langevin Dynamics (SR-LD)** on the pretrained CIFAR-10
unconditional EBM (`cifar10_large_model_uncond`, iter 121200) for 2,000 steps in a
single chain. We then classify every sample with a pretrained CIFAR-10 ResNet20 to
measure how many of the 10 classes (modes) are visited.

**Requirements**
- Pretrained TF checkpoint at `sandbox_cachedir/cachedir/cifar10_large_model_uncond/model_121200.*`
- TensorFlow 1.12 environment (see `requirements.txt`)
- `chenyaofo/pytorch-cifar-models` downloaded once via `torch.hub`

All randomness (NumPy, PyTorch, TF, cuDNN) is seeded for reproducibility.


## 1. Imports, Flags, and Global Seeds

In [ ]:
import os
import os.path as osp
import sys
import random
import json as _json

import numpy as np
import tensorflow as tf
from tensorflow.python.platform import flags
from absl import flags as absl_flags
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from models import ResNet32, ResNet32Large, ResNet32Larger, ResNet32Wider, ResNet128
from utils import optimistic_remap_restore
from baselines.common.tf_util import initialize

# ---------------------------------------------------------------
# Reproducibility: fix all seeds
# ---------------------------------------------------------------
SEED = 22
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
tf.set_random_seed(SEED)


# ---------------------------------------------------------------
# Flags (safe against duplicate re-definition in Jupyter)
# ---------------------------------------------------------------
def safe_define(define_func, name, default, help_str):
    try:
        define_func(name, default, help_str)
    except absl_flags.DuplicateFlagError:
        pass

safe_define(flags.DEFINE_string,  "dataset",            "cifar10",                   "dataset")
safe_define(flags.DEFINE_string,  "logdir",             "/home2/geeho/ebm_code_release/sandbox_cachedir/cachedir", "checkpoint root")
safe_define(flags.DEFINE_string,  "exp",                "cifar10_large_model_uncond","experiment name")
safe_define(flags.DEFINE_bool,    "cclass",             False,                       "class-conditional")
safe_define(flags.DEFINE_bool,    "bn",                 False,                       "use batchnorm")
safe_define(flags.DEFINE_bool,    "spec_norm",          True,                        "use spectral norm")
safe_define(flags.DEFINE_bool,    "use_bias",           True,                        "bias in conv")
safe_define(flags.DEFINE_bool,    "use_attention",      False,                       "self-attention")
safe_define(flags.DEFINE_float,   "step_lr",            9.0,                         "Langevin step size")
safe_define(flags.DEFINE_integer, "num_steps",          450,                         "default num steps")
safe_define(flags.DEFINE_float,   "proj_norm",          1.0,                        "grad clip")
safe_define(flags.DEFINE_integer, "batch_size",         128,                         "default batch")
safe_define(flags.DEFINE_integer, "resume_iter",        121200,                      "checkpoint iter")
safe_define(flags.DEFINE_integer, "ensemble",           1,                           "# ensemble models")
safe_define(flags.DEFINE_float,   "noise_scale",        0.005,                       "Langevin noise std")
safe_define(flags.DEFINE_bool,    "large_model",        True,                        "use large EBM")
safe_define(flags.DEFINE_bool,    "larger_model",       False,                       "use larger EBM")
safe_define(flags.DEFINE_bool,    "wider_model",        False,                       "use wider EBM")
safe_define(flags.DEFINE_integer, "no_mix",             30,                          "")
safe_define(flags.DEFINE_float,   "repellent_alpha",    0.01,                        "SR alpha")
safe_define(flags.DEFINE_float,   "repellent_c",        1e-3,                        "SR finite-diff bandwidth")
safe_define(flags.DEFINE_float,   "repellent_gamma_c",  1.0,                         "SR gamma schedule c")
safe_define(flags.DEFINE_float,   "repellent_gamma_rho",1.0,                         "SR gamma schedule rho")
safe_define(flags.DEFINE_integer, "total_chain_steps",  2000,                        "steps for single chain")
safe_define(flags.DEFINE_string,  "output_dir",         "mode_coverage_results",     "output dir")
safe_define(flags.DEFINE_integer, "seed",               SEED,                        "random seed")

FLAGS = flags.FLAGS
try:
    FLAGS(sys.argv[:1])
except absl_flags.DuplicateFlagError:
    pass

os.makedirs(FLAGS.output_dir, exist_ok=True)

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

print(f"Seed: {SEED}")
print(f"Checkpoint: {FLAGS.logdir}/{FLAGS.exp}/model_{FLAGS.resume_iter}")


/home/geeho/anaconda3/envs/ebm/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:523: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/geeho/anaconda3/envs/ebm/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:524: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/geeho/anaconda3/envs/ebm/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:525: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/geeho/anaconda3/envs/ebm/lib/python3.6/site-packages

Seed: 42
Checkpoint: /home2/geeho/ebm_code_release/sandbox_cachedir/cachedir/cifar10_large_model_uncond/model_121200


## 2. Pretrained CIFAR-10 Classifier (for mode labelling)

In [2]:
class CIFAR10Classifier:
    """Wraps a pretrained CIFAR-10 ResNet20 for predict/predict_batch on numpy arrays."""
    CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
    CIFAR_STD  = [0.2023, 0.1994, 0.2010]

    def __init__(self, model):
        self.device = torch.device("cpu")   # classifier is small; CPU avoids TF/PyTorch GPU clashes
        self.model  = model.to(self.device).eval()

    def predict(self, x):
        x_torch = torch.from_numpy(x).permute(0, 3, 1, 2).float()
        mean = torch.tensor(self.CIFAR_MEAN).view(1, 3, 1, 1)
        std  = torch.tensor(self.CIFAR_STD).view(1, 3, 1, 1)
        x_torch = ((x_torch - mean) / std).to(self.device)
        with torch.no_grad():
            logits = self.model(x_torch)
            probs  = F.softmax(logits, dim=1)
            preds  = torch.argmax(logits, dim=1)
        return preds.cpu().numpy(), probs.cpu().numpy()

    def predict_batch(self, x, batch_size=128):
        preds_all, confs_all = [], []
        for i in range(0, len(x), batch_size):
            preds, probs = self.predict(x[i:i + batch_size])
            preds_all.append(preds)
            confs_all.append(np.max(probs, axis=1))
        return np.concatenate(preds_all), np.concatenate(confs_all)


print("Loading pretrained CIFAR-10 ResNet20 classifier...")
_clf_model = torch.hub.load("chenyaofo/pytorch-cifar-models",
                            "cifar10_resnet20", pretrained=True)
classifier = CIFAR10Classifier(_clf_model)
print("Classifier ready.")


Loading pretrained CIFAR-10 ResNet20 classifier...
Classifier ready.


Using cache found in /home/geeho/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


## 3. Build the EBM, Load Weights, Create SR-LD Sampling Ops

`create_sampling_ops` here only builds the **SR-LD** graph (no vanilla Langevin branch).


In [ ]:
def build_model_from_flags(FLAGS):
    if FLAGS.dataset == "imagenetfull":
        return ResNet128(num_filters=64)
    if FLAGS.large_model:
        return ResNet32Large(num_filters=128)
    if FLAGS.larger_model:
        return ResNet32Larger(num_filters=FLAGS.hidden_dim)
    if FLAGS.wider_model:
        return ResNet32Wider(num_filters=256, train=False)
    return ResNet32(num_filters=128)


def create_sr_sampling_ops(model, weight, FLAGS):
    """Score-Repellent Langevin op (SR-LD only)."""
    X           = tf.placeholder(tf.float32, [None, 32, 32, 3], name="X")
    Y           = tf.placeholder(tf.float32, [None, 10],         name="Y")
    NOISE_SCALE = tf.placeholder(tf.float32, [1],                name="NOISE_SCALE")
    THETA       = tf.placeholder(tf.float32, [None, 32, 32, 3],  name="THETA")
    STEP        = tf.placeholder(tf.int32,   [],                 name="STEP")

    X_noisy = X + tf.random_normal(tf.shape(X), stddev=FLAGS.noise_scale * NOISE_SCALE)

    alpha = FLAGS.repellent_alpha
    c     = FLAGS.repellent_c

    energy_x = model.forward(X_noisy, weight, label=Y, reuse=True)
    grad_x   = tf.gradients(energy_x, [X_noisy])[0]
    score_x  = -grad_x

    X_pert      = X_noisy + c * THETA
    energy_pert = model.forward(X_pert, weight, label=Y, reuse=True)
    grad_pert   = tf.gradients(energy_pert, [X_pert])[0]
    H_theta     = (grad_pert - grad_x) / c

    score_surr = score_x + alpha * H_theta
    # if FLAGS.proj_norm > 0:
    #     score_surr = tf.clip_by_value(score_surr, -FLAGS.proj_norm, FLAGS.proj_norm)

    # X_repellent = tf.clip_by_value(X_noisy + FLAGS.step_lr * score_surr, 0, 1)
    X_repellent = X_noisy + FLAGS.step_lr * score_surr

    energy_new = model.forward(X_repellent, weight, label=Y, reuse=True)
    grad_new   = tf.gradients(energy_new, [X_repellent])[0]
    score_new  = -grad_new

    gamma = tf.maximum(
        FLAGS.repellent_gamma_c / tf.pow(tf.cast(STEP + 1, tf.float32), FLAGS.repellent_gamma_rho),
        0.6,
    )
    THETA_new = THETA + gamma * (score_new - THETA)

    return dict(X=X, Y=Y, NOISE_SCALE=NOISE_SCALE, THETA=THETA, STEP=STEP,
                X_repellent=X_repellent, THETA_new=THETA_new, energy=energy_x)


In [4]:
# TF session
config = tf.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.InteractiveSession(config=config)
tf.set_random_seed(SEED)

# Build EBM and load weights
model = build_model_from_flags(FLAGS)
logdir = osp.join(FLAGS.logdir, FLAGS.exp)
model_list = [FLAGS.resume_iter - 300 * i for i in range(FLAGS.ensemble)]

weights = []
for i, model_num in enumerate(model_list):
    scope  = f"context_{i}"
    weight = model.construct_weights(scope)
    initialize()

    save_file = osp.join(logdir, f"model_{model_num}")
    if not osp.exists(save_file + ".index"):
        raise FileNotFoundError(
            f"Checkpoint not found: {save_file}.index\n"
            f"Resolved logdir: {logdir}\n"
            f"See README for where to download the pretrained EBM."
        )

    v_list = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope=scope)
    v_map  = {(v.name.replace(scope, "context_0")[:-2]): v for v in v_list}
    saver  = tf.train.Saver(v_map)
    try:
        saver.restore(sess, save_file)
        print(f"[OK] Loaded {save_file}")
    except Exception as e:
        print(f"[WARN] saver.restore failed ({e}); falling back to optimistic_remap_restore")
        optimistic_remap_restore(sess, save_file, i)
    weights.append(weight)

ops = create_sr_sampling_ops(model, weights[0], FLAGS)


INFO:tensorflow:Restoring parameters from /home2/geeho/ebm_code_release/sandbox_cachedir/cachedir/cifar10_large_model_uncond/model_121200
[WARN] saver.restore failed (Restoring from checkpoint failed. This is most likely due to a Variable name or other graph key that is missing from the checkpoint. Please ensure that you have not altered the graph expected based on the checkpoint. Original error:

Key context_0/c1_pre/cb not found in checkpoint
	 [[node save/RestoreV2 (defined at <ipython-input-4-6fbd8179adac>:28)  = RestoreV2[dtypes=[DT_FLOAT, DT_FLOAT, DT_FLOAT, DT_FLOAT, DT_FLOAT, ..., DT_FLOAT, DT_FLOAT, DT_FLOAT, DT_FLOAT, DT_FLOAT], _device="/job:localhost/replica:0/task:0/device:CPU:0"](_arg_save/Const_0_0, save/RestoreV2/tensor_names, save/RestoreV2/shape_and_slices)]]

Caused by op 'save/RestoreV2', defined at:
  File "/home/geeho/anaconda3/envs/ebm/lib/python3.6/runpy.py", line 193, in _run_module_as_main
    "__main__", mod_spec)
  File "/home/geeho/anaconda3/envs/ebm/lib/py

## 4. Helpers: Mode-Coverage Metrics

In [5]:
def compute_mode_coverage_metrics(class_counts):
    """Return modes_covered, KL to uniform, TV dist, and normalised entropy."""
    total = np.sum(class_counts)
    p = class_counts / total
    q = np.ones(10) / 10
    modes_covered = int(np.sum(class_counts > 0))
    kl = 0.0
    for i in range(10):
        if p[i] > 0:
            kl += p[i] * np.log(p[i] / q[i])
    tv = 0.5 * np.sum(np.abs(p - q))
    entropy = -np.sum(p[p > 0] * np.log(p[p > 0]))
    norm_ent = entropy / np.log(10)
    return dict(modes_covered=modes_covered, kl_div=float(kl),
                tv_dist=float(tv), norm_entropy=float(norm_ent),
                class_counts=class_counts.tolist())


## 5. Run SR-LD on a Single Chain

In [ ]:
# Re-seed right before sampling so the run is deterministic
np.random.seed(SEED)
random.seed(SEED)
tf.set_random_seed(SEED)

out_dir = osp.join(FLAGS.output_dir, "exp2_single_chain")
os.makedirs(out_dir, exist_ok=True)

total_steps  = FLAGS.total_chain_steps   # 2000
noise_scale  = np.array([1], dtype=np.float32)
identity     = np.eye(10, dtype=np.float32)

x_init = np.random.uniform(0, 1, (1, 32, 32, 3)).astype(np.float32)

print("Running SR-LD single chain...")
print(f"Total steps: {total_steps}")

x     = x_init.copy()
theta = np.zeros_like(x_init)
sr_samples = []

for step in tqdm(range(total_steps), desc="SR-LD"):
    label = identity[np.random.randint(10)].reshape(1, 10)
    x, theta = sess.run(
        [ops["X_repellent"], ops["THETA_new"]],
        feed_dict={
            ops["X"]: x, ops["Y"]: label,
            ops["NOISE_SCALE"]: noise_scale,
            ops["THETA"]: theta, ops["STEP"]: step,
        },
    )
    sr_samples.append(x[0].copy())

sr_samples = np.array(sr_samples)

print(f"Classifying {len(sr_samples)} samples...")
sr_preds, sr_confs = classifier.predict_batch(sr_samples, batch_size=128)
sr_counts  = np.bincount(sr_preds, minlength=10)
sr_metrics = compute_mode_coverage_metrics(sr_counts)
sr_metrics["mean_confidence"] = float(np.mean(sr_confs))

print("\nSR-LD Results:")
print(f"  Modes covered: {sr_metrics['modes_covered']}/10")
print(f"  KL divergence: {sr_metrics['kl_div']:.4f}")
print(f"  TV distance:   {sr_metrics['tv_dist']:.4f}")
print(f"  Norm entropy:  {sr_metrics['norm_entropy']:.4f}")
print(f"  Distribution:  {sr_counts}")

with open(osp.join(out_dir, "metrics.json"), "w") as f:
    _json.dump(sr_metrics, f, indent=2)
np.save(osp.join(out_dir, "sr_preds.npy"), sr_preds)
np.save(osp.join(out_dir, "sr_confs.npy"), sr_confs)
print(f"Saved metrics and predictions to {out_dir}")


Running SR-LD single chain...
Total steps: 2000


SR-LD: 100%|██████████| 2000/2000 [01:22<00:00, 24.37it/s]


Classifying 2000 samples...

SR-LD Results:
  Modes covered: 8/10
  KL divergence: 0.4869
  TV distance:   0.4250
  Norm entropy:  0.7885
  Distribution:  [444 314  74   0  98  26 426 152   0 466]
Saved metrics and predictions to mode_coverage_results/exp2_single_chain


## 6. Visualisation — Single-Chain SR-LD Coverage

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_pos = np.arange(10)

# Class distribution
axes[0].bar(x_pos, sr_counts, color="darkorange", alpha=0.85, label="SR-LD")
axes[0].axhline(y=total_steps/10, color="red", linestyle="--", label="Uniform")
axes[0].set_ylabel("Count", fontsize=14)
axes[0].set_title("Class Distribution (Single Chain)", fontsize=15, fontweight="bold")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(axis="y", alpha=0.3)

# Metrics
metric_names = ["KL Div", "TV Dist", "1-Entropy"]
r_vals = [sr_metrics["kl_div"], sr_metrics["tv_dist"], 1 - sr_metrics["norm_entropy"]]
axes[1].bar(np.arange(3), r_vals, color="darkorange", alpha=0.85, label="SR-LD")
axes[1].set_ylabel("Value (lower is better)", fontsize=14)
axes[1].set_title("Mode Coverage Metrics", fontsize=15, fontweight="bold")
axes[1].set_xticks(np.arange(3))
axes[1].set_xticklabels(metric_names, fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(axis="y", alpha=0.3)

# Unique classes over time
seen, cum = set(), []
for p in sr_preds:
    seen.add(int(p))
    cum.append(len(seen))
axes[2].plot(cum, color="darkorange", linewidth=2.5, label="SR-LD")
axes[2].axhline(y=10, color="red", linestyle="--", linewidth=2, label="All modes")
axes[2].set_xlabel("Step", fontsize=14)
axes[2].set_ylabel("Cumulative Unique Classes", fontsize=14)
axes[2].set_title("Mode Discovery Over Time", fontsize=15, fontweight="bold")
axes[2].set_ylim(0, 11)
axes[2].legend(fontsize=11)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(osp.join(out_dir, "single_chain_coverage.pdf"), dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved to {osp.join(out_dir, 'single_chain_coverage.pdf')}")
